[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

© Pythonista · Licencia CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/deed.es

&copy; José Luis Chiquete Valdivieso. 2017-2026.

# API en la red local y control de acceso.

**py521 · IA Generativa en Entornos Locales** · Pythonista® · Instructor: José Luis Chiquete · Demo del Tema 8: *Despliegue*

En este *notebook* se consultará el modelo desde otra aplicación a través de la API HTTP compatible con *OpenAI*, se comprobará quién puede acceder al servicio y se limitará ese acceso con la dirección de escucha, un proxy con clave y el cortafuegos.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* El servicio se elige con `LLM_BASE_URL`; el proxy de la sección 4 usa `LLM_PROXY_HOST`, `LLM_PROXY_PORT` y la clave `LLM_PROXY_KEY`.
* Si `LLM_PROXY_KEY` no está definida, el *notebook* genera una clave temporal en memoria y **no la muestra**. Para crear una permanente, usa la sección 3.2 de `00_preliminares_entorno_y_secretos.ipynb`.
* Nada en este *notebook* abre el servicio a la red: todo escucha en `127.0.0.1` salvo que tú cambies `LLM_PROXY_HOST`.

### Configuración común.

Esta celda es **idéntica en todos los *notebooks***: lee la configuración de variables de entorno (o del archivo `.env`, si existe) y usa valores por omisión cuando no están definidas. Muestra todas las variables, pero **en esta demo importan: `LLM_BASE_URL`, `LLM_MODEL`, `LLM_API_KEY`, `LLM_PROXY_KEY`, `LLM_PROXY_HOST` y `LLM_PROXY_PORT`**. Las claves solo se muestran como «definida / no definida». La tabla completa de variables está en el `README.md` y en `00_preliminares_entorno_y_secretos.ipynb`.

In [ ]:
# Configuración común de py521 (celda idéntica en todos los notebooks).
import os
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)  # las variables del sistema tienen prioridad sobre .env

def _entorno(nombre, por_omision=None):
    valor = os.environ.get(nombre, "").strip()
    return valor or por_omision

# Servicio compatible con OpenAI (Ollama, LM Studio, llama.cpp server, vLLM...).
LLM_BASE_URL = _entorno("LLM_BASE_URL", _entorno("LLM_DEFAULT_BASE_URL", "http://localhost:11434/v1")).rstrip("/")  # el contenedor fija LLM_DEFAULT_BASE_URL
LLM_API_KEY = _entorno("LLM_API_KEY")              # secreto: sin valor por omisión
LLM_MODEL = _entorno("LLM_MODEL", "qwen3:1.7b")
LLM_EMBED_MODEL = _entorno("LLM_EMBED_MODEL", "qwen3-embedding:0.6b")
LLM_MODEL_A = _entorno("LLM_MODEL_A", "qwen3:1.7b-q4_K_M")
LLM_MODEL_B = _entorno("LLM_MODEL_B", "qwen3:1.7b-q8_0")
LLM_CONTEXT_A = int(_entorno("LLM_CONTEXT_A", "4096"))
LLM_CONTEXT_B = int(_entorno("LLM_CONTEXT_B", "16384"))
RAG_DOCS_DIR = Path(_entorno("RAG_DOCS_DIR", "data"))
LLM_PROXY_KEY = _entorno("LLM_PROXY_KEY")          # secreto de la demo 05: sin valor por omisión
LLM_PROXY_HOST = _entorno("LLM_PROXY_HOST", "127.0.0.1")
LLM_PROXY_PORT = int(_entorno("LLM_PROXY_PORT", "8800"))

# API nativa de Ollama (solo para las funciones marcadas «específico de Ollama»).
# Se usa OLLAMA_HOST, como el cliente oficial de Ollama; si no existe, se deriva de LLM_BASE_URL.
def _url_ollama(valor):
    valor = (valor or LLM_BASE_URL.removesuffix("/v1")).strip().rstrip("/")
    if "://" not in valor:
        anfitrion = valor.split("/")[0]
        valor = "http://" + (valor if ":" in anfitrion.split("]")[-1] else valor.replace(anfitrion, anfitrion + ":11434", 1))
    return valor.replace("://0.0.0.0", "://localhost")  # 0.0.0.0 es una dirección de escucha, no de destino

OLLAMA_URL = _url_ollama(_entorno("OLLAMA_HOST"))

for nombre in ["LLM_BASE_URL", "LLM_MODEL", "LLM_EMBED_MODEL", "LLM_MODEL_A", "LLM_MODEL_B",
               "LLM_CONTEXT_A", "LLM_CONTEXT_B", "RAG_DOCS_DIR", "LLM_PROXY_HOST", "LLM_PROXY_PORT", "OLLAMA_URL"]:
    print(f"{nombre:<16} {globals()[nombre]}")
for nombre in ["LLM_API_KEY", "LLM_PROXY_KEY", "HF_TOKEN"]:
    print(f"{nombre:<16} {'definida' if _entorno(nombre) else 'no definida'}")

### Preparación del entorno.

* Las siguientes celdas crearán el cliente de la API compatible con *OpenAI*, comprobarán que los modelos están disponibles y definirán las funciones de la demo; no hace falta leerlas.

In [ ]:
from openai import OpenAI

cliente = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY or "sin-clave")  # Ollama y LM Studio sin autenticación ignoran la clave

def comprobar_servicio(modelos):
    """Comprueba que el servicio responde (GET /models) y que ofrece los modelos indicados."""
    try:
        disponibles = {m.id for m in cliente.models.list().data}
    except Exception as error:
        raise RuntimeError(f"No responde el servicio en {LLM_BASE_URL} ({type(error).__name__}). "
                           "Abre Ollama o el motor que uses, o revisa LLM_BASE_URL.") from None
    faltan = [m for m in modelos if m not in disponibles and f"{m}:latest" not in disponibles]
    if faltan:
        raise RuntimeError(f"El servicio no ofrece: {', '.join(faltan)}. En Ollama: "
                           + " · ".join(f"ollama pull {m}" for m in faltan))
    print(f"Servicio en {LLM_BASE_URL}: responde. Modelos: {', '.join(modelos)}.")

comprobar_servicio([LLM_MODEL])

In [ ]:
import hmac
import json
import secrets
import socket
import threading
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer
from urllib.parse import urlsplit

import httpx

## 1. Consultar el modelo desde otra aplicación.

* La API compatible con *OpenAI* es HTTP: cualquier aplicación que pueda enviar una petición HTTP puede usar el modelo, sin importar el lenguaje.
* La siguiente celda hará la petición «a mano», como lo haría otra aplicación: `GET /models` para listar modelos y `POST /chat/completions` para conversar.
* En este *notebook*, «otra aplicación» se ejecuta **en el mismo equipo** que el modelo. La prueba desde un segundo dispositivo de la red es opcional y está en la sección 5.1.

In [ ]:
cabeceras = {"Authorization": f"Bearer {LLM_API_KEY}"} if LLM_API_KEY else {}

modelos = httpx.get(f"{LLM_BASE_URL}/models", headers=cabeceras, timeout=10).json()
print("GET /models →", [m["id"] for m in modelos["data"]])

cuerpo = {"model": LLM_MODEL, "temperature": 0, "max_tokens": 80, "reasoning_effort": "none",
          "messages": [{"role": "user", "content": "Explica en una frase qué es una API."}]}
respuesta = httpx.post(f"{LLM_BASE_URL}/chat/completions", json=cuerpo, headers=cabeceras, timeout=300)
print("POST /chat/completions →", respuesta.status_code)
print(respuesta.json()["choices"][0]["message"]["content"])

* Desde otro equipo de la red, la misma petición usa la dirección de ese servidor en lugar de `localhost`. Por ejemplo, con *curl*:

```bash
curl http://<ip-del-servidor>:8800/v1/chat/completions \
  -H "Authorization: Bearer $LLM_API_KEY" -H "Content-Type: application/json" \
  -d '{"model": "qwen3:1.7b", "messages": [{"role": "user", "content": "Hola"}]}'
```

## 2. Quién puede acceder.

### 2.1 La dirección de escucha decide quién llega.

| Dirección de escucha | Quién puede conectarse | Uso |
|---|---|---|
| `127.0.0.1` (*localhost*) | Solo programas de este equipo | Valor por omisión de *Ollama*, *LM Studio* y *llama.cpp server*. Seguro para uso personal |
| IP concreta de la red local (p. ej. `192.168.1.10`) | Equipos que alcanzan esa IP | Solo con cortafuegos que limite los clientes y con clave |
| `0.0.0.0` (todas las interfaces) | Cualquiera que alcance el equipo por cualquier red | ⚠️ No recomendable sin cortafuegos y proxy con clave |

La siguiente celda buscará la IP de este equipo en la red local y probará si el puerto del servicio responde ahí.

In [ ]:
def ip_local():
    """IP de este equipo en la red local (no envía paquetes)."""
    with socket.socket(socket.AF_INET, socket.SOCK_DGRAM) as s:
        try:
            s.connect(("192.0.2.1", 80))  # dirección de documentación; solo elige la interfaz de salida
            return s.getsockname()[0]
        except OSError:
            return None

def puerto_abierto(anfitrion, puerto, espera=2):
    try:
        socket.create_connection((anfitrion, puerto), timeout=espera).close()
        return True
    except OSError:
        return False

servicio = urlsplit(LLM_BASE_URL)
puerto_servicio = servicio.port or (443 if servicio.scheme == "https" else 80)
mi_ip = ip_local()
print(f"Servicio: {servicio.hostname}:{puerto_servicio} · IP de este equipo en la red local: {mi_ip or 'sin red'}")
if servicio.hostname in ("localhost", "127.0.0.1", "::1") and mi_ip:
    visible = puerto_abierto(mi_ip, puerto_servicio)
    print(f"¿Responde el puerto {puerto_servicio} en {mi_ip}?", "SÍ ⚠️ otros equipos de la red podrían usarlo" if visible
          else "no ✅ solo escucha en este equipo")
else:
    print("El servicio está en otro equipo o contenedor; revisa su dirección de escucha en ese equipo.")

### 2.2 Riesgos de abrir el servicio sin protección.

* **La API de *Ollama* no pide autenticación.** Quien alcance el puerto puede generar texto con tus recursos, listar tus modelos y también descargar o borrar modelos con la API nativa (`/api/pull`, `/api/delete`) ([referencia de la API](https://docs.ollama.com/api/introduction)).
* Por eso **no se recomienda `OLLAMA_HOST=0.0.0.0`** sin cortafuegos ni proxy con clave: deja el servicio abierto a toda red a la que esté conectado el equipo, incluidas redes públicas de wifi.
* *LM Studio* tampoco pide clave por omisión; al activar **Serve on Local Network**, su documentación recomienda activar **Require Authentication** ([documentación](https://lmstudio.ai/docs/developer/core/server/serve-on-network)).
* *llama.cpp server* admite `--api-key` ([README](https://github.com/ggml-org/llama.cpp/blob/master/tools/server/README.md)). En *vLLM*, `--api-key` protege solo algunas rutas; su documentación recomienda un proxy inverso ([documentación](https://docs.vllm.ai/en/latest/serving/online_serving/openai_compatible_server.html)).
* Una clave sin HTTPS viaja en texto claro por la red: sirve en una red local de confianza, no en internet.

## 3. Cómo limitar el acceso.

La opción segura combina tres capas:

1. **Dirección de escucha:** el motor sigue en `127.0.0.1`.
2. **Proxy con clave:** un programa intermedio recibe las peticiones, exige la clave, deja pasar solo las rutas necesarias y las reenvía al motor.
3. **Cortafuegos:** solo los equipos autorizados llegan al puerto del proxy.

## 4. Proxy con clave.

* El proxy de la siguiente celda es **didáctico**: muestra la idea en pocas líneas, sin HTTPS. En un servicio real se usa un proxy inverso como *Caddy* o *nginx* con HTTPS.
* Exige `Authorization: Bearer <clave>`, compara la clave en tiempo constante (`hmac.compare_digest`) y solo deja pasar `GET /v1/models`, `POST /v1/chat/completions` y `POST /v1/embeddings`.
* Registra cada petición (origen, ruta y resultado) sin guardar la clave.
* Para repetir, ejecuta antes la celda que detiene el proxy (final de la sección 5). Si no lo haces, la siguiente celda lo detiene antes de crear uno nuevo.

In [ ]:
if "proxy" in globals():  # proxy de una ejecución anterior: se detiene para liberar el puerto
    proxy.shutdown()
    proxy.server_close()

CLAVE_PROXY = LLM_PROXY_KEY or secrets.token_urlsafe(32)  # temporal si no está definida; nunca se imprime
print("Clave del proxy:", "LLM_PROXY_KEY (definida)" if LLM_PROXY_KEY else "temporal en memoria (no se muestra)")

RUTAS_PERMITIDAS = {("GET", "/v1/models"), ("POST", "/v1/chat/completions"), ("POST", "/v1/embeddings")}
registro = []

class ProxyConClave(BaseHTTPRequestHandler):
    def _atender(self, metodo):
        ruta = urlsplit(self.path).path.rstrip("/")
        autorizacion = self.headers.get("Authorization", "")
        if not hmac.compare_digest(autorizacion.encode(), f"Bearer {CLAVE_PROXY}".encode()):
            return self._responder(401, {"error": "clave ausente o incorrecta"}, ruta)
        if (metodo, ruta) not in RUTAS_PERMITIDAS:
            return self._responder(403, {"error": "ruta no permitida"}, ruta)
        cuerpo = self.rfile.read(int(self.headers.get("Content-Length", 0)))
        destino = LLM_BASE_URL + ruta.removeprefix("/v1")
        reenvio = httpx.request(metodo, destino, content=cuerpo or None, timeout=600,
                                headers={"Content-Type": "application/json", **cabeceras})
        self._responder(reenvio.status_code, reenvio.content, ruta, reenvio.headers.get("Content-Type"))

    def _responder(self, estado, contenido, ruta, tipo="application/json"):
        datos = contenido if isinstance(contenido, bytes) else json.dumps(contenido).encode()
        self.send_response(estado)
        self.send_header("Content-Type", tipo or "application/json")
        self.send_header("Content-Length", str(len(datos)))
        self.end_headers()
        self.wfile.write(datos)
        registro.append((self.client_address[0], self.command, ruta, estado))

    def do_GET(self):
        self._atender("GET")

    def do_POST(self):
        self._atender("POST")

    def log_message(self, *args):  # el registro propio sustituye al de la consola
        pass

proxy = ThreadingHTTPServer((LLM_PROXY_HOST, LLM_PROXY_PORT), ProxyConClave)
threading.Thread(target=proxy.serve_forever, daemon=True).start()
URL_PROXY = f"http://{'localhost' if LLM_PROXY_HOST in ('127.0.0.1', '0.0.0.0') else LLM_PROXY_HOST}:{LLM_PROXY_PORT}/v1"
print(f"Proxy escuchando en {LLM_PROXY_HOST}:{LLM_PROXY_PORT} → reenvía a {LLM_BASE_URL}")

### 4.1 Pruebas de acceso.

* Sin clave, con una clave incorrecta y a una ruta no permitida (`/api/pull` de *Ollama*, que descarga modelos), el proxy rechaza la petición **antes** de llegar al motor.

In [ ]:
pruebas = [
    ("Sin clave", "GET", "/v1/models", {}),
    ("Clave incorrecta", "GET", "/v1/models", {"Authorization": "Bearer clave-incorrecta"}),
    ("Ruta no permitida", "POST", "/api/pull", {"Authorization": f"Bearer {CLAVE_PROXY}"}),
    ("Clave correcta", "GET", "/v1/models", {"Authorization": f"Bearer {CLAVE_PROXY}"}),
]
base_proxy = URL_PROXY.removesuffix("/v1")
for nombre, metodo, ruta, cab in pruebas:
    r = httpx.request(metodo, base_proxy + ruta, headers=cab, timeout=30,
                      json={"model": "x"} if metodo == "POST" else None)
    print(f"{nombre:<18} {metodo} {ruta:<16} → {r.status_code}")

### 4.2 Otra aplicación a través del proxy.

* La aplicación cliente (aquí, en el mismo equipo) solo necesita la URL del proxy y la clave: con el cliente oficial de *OpenAI* son `base_url` y `api_key`. En otro equipo serían sus variables `LLM_BASE_URL` y `LLM_API_KEY`.

In [ ]:
otra_aplicacion = OpenAI(base_url=URL_PROXY, api_key=CLAVE_PROXY)
r = otra_aplicacion.chat.completions.create(
    model=LLM_MODEL, temperature=0, max_tokens=60, reasoning_effort="none",
    messages=[{"role": "user", "content": "Responde en una frase: ¿para qué sirve una clave de API?"}])
print(r.choices[0].message.content)

### 4.3 Registro de accesos.

* El registro permite saber quién hizo qué y desde dónde.

In [ ]:
print(f"{'Origen':<16}{'Método':<8}{'Ruta':<24}Estado")
for origen, metodo, ruta, estado in registro:
    print(f"{origen:<16}{metodo:<8}{ruta:<24}{estado}")

## 5. Abrir el proxy a la red local con cortafuegos.

Para que otro equipo de la red use el modelo, solo el **proxy** se abre, nunca el motor:

1. El motor sigue en `127.0.0.1` (en *Ollama*, sin definir `OLLAMA_HOST` en el servicio).
2. Define `LLM_PROXY_HOST` con la IP de este equipo en la red local (sección 2.1) y una clave permanente en `LLM_PROXY_KEY`; reinicia el *kernel* y ejecuta de nuevo.
3. Permite en el cortafuegos el puerto del proxy **solo** desde los equipos autorizados. Ejemplos (requieren permisos de administrador; ejecútalos en una terminal, no en el *notebook*), con el cliente `192.168.1.20` y el puerto `8800`:

| Sistema | Regla |
|---|---|
| Linux (*ufw*) | `sudo ufw allow from 192.168.1.20 to any port 8800 proto tcp` |
| Windows (*PowerShell*) | `New-NetFirewallRule -DisplayName "py521 proxy" -Direction Inbound -Protocol TCP -LocalPort 8800 -RemoteAddress 192.168.1.20 -Action Allow` |
| macOS | El cortafuegos de **Ajustes del Sistema → Red → Cortafuegos** permite o bloquea por aplicación, no por IP; para limitar por IP hace falta *pf* |

4. En el otro equipo, define `LLM_BASE_URL=http://<ip-de-este-equipo>:8800/v1` y `LLM_API_KEY` con la clave. La clave se comparte por un canal seguro (por ejemplo, un gestor de contraseñas), nunca en pantalla ni en el repositorio.
5. Al terminar, detén el proxy, elimina la regla del cortafuegos y rota la clave.

### 5.1 Prueba en vivo desde un segundo dispositivo (opcional).

Requiere un segundo equipo en la misma red local con *curl* (incluido en Windows 10 y posteriores, macOS y la mayoría de distribuciones Linux).

1. En este equipo, sigue los pasos 2 y 3 anteriores y ejecuta de nuevo el *notebook* hasta la sección 4.1.
2. En el segundo equipo, define la clave en una variable de entorno (sin escribirla en pantalla compartida) y ejecuta:

```bash
curl http://<ip-de-este-equipo>:8800/v1/models -H "Authorization: Bearer $LLM_API_KEY"
```

En *PowerShell* (Windows), usa `curl.exe` (`curl` es otro comando) y `$env:LLM_API_KEY`:

```powershell
curl.exe http://<ip-de-este-equipo>:8800/v1/models -H "Authorization: Bearer $env:LLM_API_KEY"
```

3. Debe responder con la lista de modelos. Sin la cabecera `Authorization`, debe responder `401`.
4. Vuelve a ejecutar la celda de la sección 4.3: el registro muestra la IP del segundo equipo como origen.
5. Al terminar, ejecuta la siguiente celda y quita la regla del cortafuegos.

> Con el contenedor de desarrollo en Linux, *Ollama* puede escuchar en la IP del puente de *Docker* (por lo general `172.17.0.1`), que no es visible desde la red local; ver `00_preliminares_entorno_y_secretos.ipynb`.

In [ ]:
proxy.shutdown()
proxy.server_close()
print("Proxy detenido.")

---
## ✅ Cierre.

* Cualquier aplicación que hable HTTP puede usar el modelo a través de la API compatible con *OpenAI*.
* El motor escucha en `127.0.0.1`; si otros equipos lo necesitan, se abre solo un proxy con clave, limitado por cortafuegos y, fuera de una red de confianza, con HTTPS.
* `0.0.0.0` sin cortafuegos ni clave deja el modelo disponible para cualquiera que alcance el equipo.

<p style="text-align: center"><a rel="license" href="https://creativecommons.org/licenses/by/4.0/deed.es"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="https://creativecommons.org/licenses/by/4.0/deed.es">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; Pythonista · Licencia CC BY 4.0 — https://creativecommons.org/licenses/by/4.0/deed.es</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>